In [5]:
!pip install torch torchvision onnx onnxruntime --quiet

import torch
import torch.nn as nn
import torchvision.models as models
import onnx
import onnxruntime as ort
import numpy as np

In [6]:
class DriverActionClassifier(nn.Module):
    def __init__(self, backbone, num_classes=10):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(3 * 576, num_classes)

    def forward(self, x):
        # x: (B, 3, 3, 224, 224)
        B = x.shape[0]
        x = x.view(B * 3, 3, 224, 224)
        feats = self.backbone(x)          # (B*3, 576)
        feats = feats.flatten(1)
        feats = feats.view(B, -1)         # (B, 3*576)
        return self.classifier(feats)


In [8]:
from google.colab import drive
drive.mount('/content/drive')

model_path = "/content/drive/MyDrive/driver_action_deploy.pth"


device = "cuda" if torch.cuda.is_available() else "cpu"

# Recreate the model structure
mobilenet = models.mobilenet_v3_small(weights=None)
mobilenet.classifier = nn.Identity()  # Remove classifier head

model = DriverActionClassifier(backbone=mobilenet, num_classes=10).to(device)

# Load the weights from Drive
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("Model loaded successfully")


Mounted at /content/drive
Model loaded successfully


In [9]:
dummy_input = torch.randn(
    1,   # batch size
    3,   # views: full, face, hand
    3,   # channels
    224,
    224,
    device=device
)

In [12]:
!pip install onnx onnxruntime onnxscript --quiet


onnx_file = "driver_action.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_file,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={
        "input": {0: "batch"},
        "logits": {0: "batch"}
    }
)

print(f"ONNX export complete: {onnx_file}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 kB 13.7 MB/s eta 0:00:00


/tmp/ipython-input-2216536076.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0116 02:28:57.555000 2318 torch/onnx/_internal/exporter/_compat.py:114] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `DriverActionClassifier([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `DriverActionClassifier([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 127, in call
    converted_proto = _c_api_utils.call_onnx_api(
                      ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
             ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnxscript/version_converter/__init__.py", line 122, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/onnx/version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_version(model_str, target_version)
                          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: /github/workspace/onnx/version_converter/adapters/axes_input_to_attribute.h:65: adapt: Asserti

[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 69 of general pattern rewrite rules.
ONNX export complete: driver_action.onnx


In [13]:
onnx_model = onnx.load(onnx_file)
onnx.checker.check_model(onnx_model)
print("ONNX model check passed")

# Test inference with ONNX Runtime
sess = ort.InferenceSession(onnx_file, providers=["CUDAExecutionProvider"])

x_test = np.random.randn(2, 3, 3, 224, 224).astype(np.float32)
out = sess.run(None, {"input": x_test})

print(f"ONNX Runtime output shape: {out[0].shape}")  # Should be (2, 10)
print("ONNX Runtime inference test passed ")

ONNX model check passed ✅
ONNX Runtime output shape: (2, 10)
ONNX Runtime inference test passed ✅


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


In [14]:
import pickle
CLASS_NAMES = {
    0: "safe driving",
    1: "texting - right",
    2: "talking on the phone - right",
    3: "texting - left",
    4: "talking on the phone - left",
    5: "operating the radio",
    6: "drinking",
    7: "reaching behind",
    8: "hair and makeup",
    9: "talking to passenger"
}
with open("class_mapping.pkl", "wb") as f:
    pickle.dump(CLASS_NAMES, f)

print("Step 2 complete: ONNX model and class mapping ready for TensorRT ")

Step 2 complete: ONNX model and class mapping ready for TensorRT 🚀


In [18]:
import json
with open("/content/drive/MyDrive/driver_class_map.json", "w") as f:
    json.dump(CLASS_NAMES, f)
!cp driver_action.onnx /content/drive/MyDrive/driver_action.onnx

